# Notebook 12: Baseline Retraining at 260px (Resolution-Matched Comparison)

## Objective
PneumoXNet (proposed model) trains at 260×260 resolution, but the original ResNet-50 and
ConvNeXt-Tiny baselines (Notebook 06, 07) were trained at 224×224. This creates an unfair
comparison. This notebook retrains both baselines at 260×260 so all models in the paper are
compared under identical input resolution.

## What this notebook does
1. Loads dataset at 260px (same preprocessing/augmentation as PneumoXNet)
2. Retrains ResNet-50 and ConvNeXt-Tiny with identical training config
3. Evaluates both on the test set and prints classification reports
4. Saves results to CSV for direct comparison with PneumoXNet

In [1]:
# ============================================================
# Cell 2: Import Required Libraries
# ============================================================

import time
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import (
    resnet50, ResNet50_Weights,
    convnext_tiny, ConvNeXt_Tiny_Weights
)

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Use GPU if available, otherwise fall back to CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Cell 2 : Libraries Imported Successfully")
print(f"Device : {DEVICE}")

Cell 2 : Libraries Imported Successfully
Device : cuda


## Configuration
Same hyperparameters as PneumoXNet's final training run (Notebook 09, 260px version) —
this is essential for a fair comparison. Only the model architecture changes between runs.

In [2]:
# ============================================================
# Cell 4: Configuration
# ============================================================

IMAGE_SIZE = 260          # matches PneumoXNet input resolution (was 224 in old baselines)
BATCH_SIZE = 16
EPOCHS = 40                # max epochs; early stopping will likely cut this short
LEARNING_RATE = 1e-4
NUM_CLASSES = 3
NUM_WORKERS = 0
CLASS_NAMES = ["BACTERIA", "NORMAL", "VIRUS"]

# Project paths — same structure as previous notebooks
PROJECT_ROOT = Path("/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI")
DATASET_DIR = PROJECT_ROOT / "dataset" / "processed_dataset"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Cell 4 : Configuration Set")
print(f"Image Size : {IMAGE_SIZE}px  |  Epochs : {EPOCHS}  |  Batch Size : {BATCH_SIZE}")

Cell 4 : Configuration Set
Image Size : 260px  |  Epochs : 40  |  Batch Size : 16


## Dataset Loading (260px)
Same augmentation pipeline used for PneumoXNet's 260px training run — this keeps the
data-side treatment identical across all models, isolating architecture as the only
variable being tested.